<a href="https://colab.research.google.com/github/GiovanniPioDelvecchio/NLP-Project/blob/GloVePreTrained/HumanValueRecognition_BERT_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers
!pip install datasets
!pip install torchinfo

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 22.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 KB 11.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 54.9 MB/s eta 0:00:00
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.8/462.8 KB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.0/213.0 KB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.0/132.0 KB 16.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 KB 16.2 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.24.3
    Uninstalling urllib3-1.24.3:
      Successfully uninstalled urllib3-1.24.3
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-whe

In [2]:
# Cell for the download of the datasets
!wget https://zenodo.org/record/7550385/files/arguments-training.tsv
!wget https://zenodo.org/record/7550385/files/labels-training.tsv
!wget https://zenodo.org/record/7550385/files/arguments-validation.tsv
!wget https://zenodo.org/record/7550385/files/labels-validation.tsv
!wget https://zenodo.org/record/7550385/files/arguments-test.tsv
!wget https://zenodo.org/record/7550385/files/arguments-validation-zhihu.tsv
!wget https://zenodo.org/record/7550385/files/labels-validation-zhihu.tsv

--2023-02-01 18:14:13--  https://zenodo.org/record/7550385/files/arguments-training.tsv
Resolving zenodo.org (zenodo.org)... 188.185.124.72
Connecting to zenodo.org (zenodo.org)|188.185.124.72|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1012498 (989K) [application/octet-stream]
Saving to: ‘arguments-training.tsv’

arguments-training. 100%[===================>] 988.77K   407KB/s    in 2.4s    

2023-02-01 18:14:18 (407 KB/s) - ‘arguments-training.tsv’ saved [1012498/1012498]

--2023-02-01 18:14:18--  https://zenodo.org/record/7550385/files/labels-training.tsv
Resolving zenodo.org (zenodo.org)... 188.185.124.72
Connecting to zenodo.org (zenodo.org)|188.185.124.72|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 253843 (248K) [application/octet-stream]
Saving to: ‘labels-training.tsv’

labels-training.tsv 100%[===================>] 247.89K   406KB/s    in 0.6s    

2023-02-01 18:14:20 (406 KB/s) - ‘labels-training.tsv’ saved [253843/2

In [3]:
# imports for dataset loading
import numpy as np
import random
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# torch imports
import torch
import torchtext
from torchtext.data import get_tokenizer
from torchtext.vocab import GloVe
from torch.utils.data import DataLoader
from torchtext.data.functional import to_map_style_dataset
from torch import nn
from torch.nn import functional as F
from torch.optim import Adam

# progress bar
from tqdm import tqdm
# garbage collector
import gc

# imports for evaluation
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

In [4]:
def fix_random(seed: int) -> None:
  """Fix all the possible sources of randomness.

  Args:
    seed: the seed to use. 
  """
  np.random.seed(seed)
  random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)

  torch.backends.cudnn.benchmark = False
  torch.backends.cudnn.deterministic = True

In [5]:
seed = 10
fix_random(seed)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [6]:
def huggingface_from_pandas(pandas_df):
  hf_ds = Dataset.from_pandas(pandas_df, preserve_index=False)
  hf_ds = hf_ds.remove_columns(["Argument ID", "Argument ID2"])
  hf_ds = hf_ds.map(lambda x:{"labels": [int(x[col]) for col in hf_ds.column_names if
                                      col not in ['Conclusion', 'Stance', 'Premise']]})
  label_cols = [col for col in hf_ds.column_names if col not in ['Conclusion', 'Stance', 'Premise', "labels"]]
  hf_ds = hf_ds.remove_columns(label_cols)
  return hf_ds, label_cols

In [7]:
# Dataset loading and splitting
raw_training = pd.read_csv("arguments-training.tsv", encoding='utf-8', sep='\t', header=0)
raw_training_lab = pd.read_csv("labels-training.tsv", encoding='utf-8', sep='\t', header=0)
raw_test = pd.read_csv("arguments-validation.tsv", encoding='utf-8', sep='\t', header=0)
raw_test_lab = pd.read_csv("labels-validation.tsv", encoding='utf-8', sep='\t', header=0)

train = raw_training.join(raw_training_lab,how='inner' ,lsuffix='2') # joining labels
test = raw_test.join(raw_test_lab, how='inner', lsuffix='2') # joining labels
train, val = train_test_split(train ,train_size=.80, random_state=seed) # splitting training

train_ds, label_list = huggingface_from_pandas(train)
val_ds, _ = huggingface_from_pandas(val)
test_ds, _ = huggingface_from_pandas(test)

print(train_ds[0])
print(label_list)

whole_dataset = DatasetDict()
whole_dataset["train"] = train_ds.with_format("torch")
whole_dataset["val"] = val_ds.with_format("torch")
whole_dataset["test"] = test_ds.with_format("torch")

  0%|          | 0/4314 [00:00<?, ?ex/s]

  0%|          | 0/1079 [00:00<?, ?ex/s]

  0%|          | 0/1896 [00:00<?, ?ex/s]

{'Conclusion': 'We should ban the Church of Scientology', 'Stance': 'in favor of', 'Premise': "Scientology is not a true religion it is a sect or a cult which brainwashes it's followers and makes money from them.", 'labels': [1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0]}
['Self-direction: thought', 'Self-direction: action', 'Stimulation', 'Hedonism', 'Achievement', 'Power: dominance', 'Power: resources', 'Face', 'Security: personal', 'Security: societal', 'Tradition', 'Conformity: rules', 'Conformity: interpersonal', 'Humility', 'Benevolence: caring', 'Benevolence: dependability', 'Universalism: concern', 'Universalism: nature', 'Universalism: tolerance', 'Universalism: objectivity']


In [ ]:
print(whole_dataset.keys())
print(whole_dataset["train"])

dict_keys(['train', 'val', 'test'])
Dataset({
    features: ['Conclusion', 'Stance', 'Premise', 'labels'],
    num_rows: 4314
})


In [ ]:
print(whole_dataset["train"]["labels"][0].shape)
print(whole_dataset["train"]["labels"])

In [42]:
# Pretrained GloVe setup

global_vectors = GloVe(name='6B', dim=100)

# the current choice is to give an id to each word
tokenizer = get_tokenizer("basic_english")

embeddings = global_vectors.get_vecs_by_tokens(tokenizer("Hello, How are you?"), lower_case_backup=True)

print(embeddings.shape)

.vector_cache/glove.6B.zip: 862MB [02:39, 5.41MB/s]                           
100%|█████████▉| 399999/400000 [00:14<00:00, 28234.16it/s]


torch.Size([6, 100])


In [90]:
max_words = 25
embed_len = 100
batch_size = 32

# collate function where the Premises are tokenized and embedded in batches
def vectorize_batch(batch):
    X = [elem["Premise"] for elem in batch]
    Y = [elem["labels"] for elem in batch]
    X = [tokenizer(x) for x in X]
    X = [tokens+[""] * (max_words-len(tokens))  if len(tokens)<max_words else tokens[:max_words] for tokens in X]
    X_tensor = torch.zeros(len(batch), max_words, embed_len)
    Y_tensor = torch.zeros(len(batch), Y[0].shape[0])
    for i, tokens in enumerate(X):
        X_tensor[i] = global_vectors.get_vecs_by_tokens(tokens)
        Y_tensor[i] = Y[i]
    return X_tensor, Y_tensor

train_dataset = whole_dataset["train"].remove_columns(["Stance", "Conclusion"])
val_dataset = whole_dataset["val"].remove_columns(["Stance", "Conclusion"])
test_dataset = whole_dataset["test"].remove_columns(["Stance", "Conclusion"])
print(val_dataset.shape)

# Construction of the Dataloaders for train and validation
train_loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=lambda x:tuple(y.to(device) for y in vectorize_batch(x)))
val_loader  = DataLoader(val_dataset, batch_size=batch_size, collate_fn=lambda x:tuple(y.to(device) for y in vectorize_batch(x)))
test_loader  = DataLoader(test_dataset, batch_size=batch_size, collate_fn=lambda x:tuple(y.to(device) for y in vectorize_batch(x)))

(1079, 2)


In [96]:
num_classes = 20

# Simple model to perform some tests with pytorch
class EmbeddingClassifier(nn.Module):
    def __init__(self):
        super(EmbeddingClassifier, self).__init__() 
        # Not sure about this
        #self.seq_length = batch_size
        self.input_dim = (batch_size, max_words,embed_len)
        self.num_layers = 1

        self.gru = nn.GRU(input_size = embed_len,
                          hidden_size = embed_len,
                          num_layers = 1,
                          batch_first=True, 
                          bidirectional = True)
        self.flatten = nn.Flatten(start_dim=1)
        self.linear_1 = nn.Linear(max_words*embed_len*2, 512)
        self.relu = nn.ReLU()

        self.linear_2 = nn.Linear(512,256)
        #nn.ReLU(),

        self.linear_3 = nn.Linear(256,128)
        #nn.ReLU(),

        self.linear_4 = nn.Linear(128, 64)
        self.linear_5 = nn.Linear(64, num_classes)
        
                

    def forward(self, X_batch):
        h0 = torch.zeros(2,X_batch.shape[0], embed_len)
        h0 = h0.to(device)
        print(X_batch.shape)
        out, hn = self.gru(X_batch, h0)
        out = self.flatten(out)
        out = self.linear_1(out)
        out = self.relu(out)
        out = self.linear_2(out)
        out = self.relu(out)
        out = self.linear_3(out)
        out = self.relu(out)
        out = self.linear_4(out)
        out = self.relu(out)
        out = self.linear_5(out)
        return out

# Function needed to compute the validation loss and the accuracy
def CalcValLossAndAccuracy(model, loss_fn, val_loader):
    with torch.no_grad():
      Y_shuffled, Y_preds, losses = [],[],[]
      for X, Y in val_loader:
        preds = model(X)
        loss = loss_fn(preds, Y)
        losses.append(loss.item())
        Y_shuffled.append(Y)
        Y_preds.append(preds.argmax(dim=-1))

      Y_shuffled = torch.cat(Y_shuffled)
      Y_preds = torch.cat(Y_preds)

      loss = torch.tensor(losses).mean()
      print("Valid Loss : {:.3f}".format(loss))
    return loss
    # print("Valid Acc  : {:.3f}".format(accuracy_score(Y_shuffled.detach().cpu().numpy(), Y_preds.detach().cpu().numpy())))

# Training function
def TrainModel(model, loss_fn, optimizer, train_loader, val_loader, epochs, early_stopping_info):
    patience_acc = 0
    precedent_loss = np.Inf
    model.train()
    for i in range(1, epochs+1):
        losses = []
        for X, Y in tqdm(train_loader):
            Y_preds = model(X)

            loss = loss_fn(Y_preds, Y)
            losses.append(loss.item())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        loss = CalcValLossAndAccuracy(model, loss_fn, val_loader)
        if precedent_loss - loss < early_stopping_info["delta"]:
           patience_acc = patience_acc + 1
        else:
          patience_acc = 0
          torch.save(model, "best.pth")

        if patience_acc > early_stopping_info["patience"]:
          return torch.load("best.pth")
        precedent_loss = loss

        if i%1==0:
            print("Train Loss : {:.3f}".format(torch.tensor(losses).mean()))
    return model

In [97]:
from torchinfo import summary
epochs = 50
learning_rate = 1e-4

loss_fn = nn.BCEWithLogitsLoss()
embed_classifier = EmbeddingClassifier()
optimizer = Adam(embed_classifier.parameters(), lr=learning_rate)

embed_classifier.to(device)
summary(embed_classifier, 
                input_size=(1,max_words, embed_len), device="cuda")
#TrainModel(embed_classifier, loss_fn, optimizer, train_loader, val_loader, epochs)

torch.Size([1, 25, 100])


Layer (type:depth-idx)                   Output Shape              Param #
EmbeddingClassifier                      [1, 20]                   --
├─GRU: 1-1                               [1, 25, 200]              121,200
├─Flatten: 1-2                           [1, 5000]                 --
├─Linear: 1-3                            [1, 512]                  2,560,512
├─ReLU: 1-4                              [1, 512]                  --
├─Linear: 1-5                            [1, 256]                  131,328
├─ReLU: 1-6                              [1, 256]                  --
├─Linear: 1-7                            [1, 128]                  32,896
├─ReLU: 1-8                              [1, 128]                  --
├─Linear: 1-9                            [1, 64]                   8,256
├─ReLU: 1-10                             [1, 64]                   --
├─Linear: 1-11                           [1, 20]                   1,300
Total params: 2,855,492
Trainable params: 2,855,492
Non-tr

In [98]:
fix_random(seed)
embed_classifier = TrainModel(embed_classifier, loss_fn, optimizer, train_loader, val_loader, epochs, {"patience": 3, "delta": 1e-4})

  5%|▌         | 7/135 [00:00<00:01, 64.67it/s]

torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])


 16%|█▋        | 22/135 [00:00<00:01, 67.46it/s]

torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])


 27%|██▋       | 36/135 [00:00<00:01, 66.84it/s]

...

 48%|████▊     | 65/135 [00:00<00:01, 65.52it/s]

torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])


 61%|██████    | 82/135 [00:01<00:00, 65.81it/s]

torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])
torch.Size([32, 25, 100])


KeyboardInterrupt: ignored

In [8]:
from os import supports_effective_ids
def make_predictions(model, loader):
    Y_shuffled, Y_preds = [], []
    model.eval()
    for X, Y in loader:
        preds = model(X)
        Y_preds.append(preds)
    gc.collect()
    Y_preds = torch.cat(Y_preds)
    Y_preds = Y_preds.sigmoid()
    return Y_preds.detach()

def keep_above_thresh(Y_preds, thr):
  Y_preds_thr = np.copy(Y_preds.numpy())
  max_rows = Y_preds_thr.shape[0]
  max_cols = Y_preds_thr.shape[1]
  for i in range(max_rows):
    new_row = np.array([1 if Y_preds_thr[i][j] > thr else 0 for j in range(max_cols)])
    Y_preds_thr[i] = new_row
  return Y_preds_thr

def compute_macro_score(M_true, M_pred, score_func):
    scores = []
    for i in range(M_true.shape[1]):
        true = M_true[:, i]
        pred = M_pred[:, i]
        if score_func == accuracy_score:
          scores.append(score_func(true, pred))
        else: 
          scores.append(score_func(true, pred, zero_division=0))
    return np.mean(scores), scores
  
def support(true, pred, zero_division):
  return sum(true)

def print_report(classifier, loader, y_true, threshold, labels=label_list):
  Y_preds = make_predictions(classifier, loader)
  Y_preds_thr = keep_above_thresh(Y_preds.to('cpu'), threshold)
  f1_macro, f1 = compute_macro_score(y_true, Y_preds_thr, f1_score)
  acc_macro, acc = compute_macro_score(y_true, Y_preds_thr, accuracy_score)
  prec_macro, prec = compute_macro_score(y_true, Y_preds_thr, precision_score)
  rec_macro, rec = compute_macro_score(y_true, Y_preds_thr, recall_score)
  _, sup = compute_macro_score(y_true, Y_preds_thr, support)
  print("----- MACRO AVG. -----")
  print(f"  F1-score:\t{round(f1_macro,4)}\n\
  Precision:\t{round(prec_macro,4)}\n\
  Recall:\t{round(rec_macro,4)}\n\
  Accuracy:\t{round(acc_macro,4)}")
  print("----- PER-CLASS VALUES -----")
  print("  \t\t\t\tF1-score\tPrecision\tRecall\t\tAccuracy\tSupport")
  for i in range(len(labels)):
    print("  " + labels[i]+" "*(len(max(labels, key=len))-len(labels[i])), end="\t")
    print(f"{round(f1[i],4)}\t\t{round(prec[i],4)}\t\t{round(rec[i],4)}\t\t{round(acc[i],4)}\t\t{sup[i]}")

In [ ]:
print_report(embed_classifier, val_loader,val_dataset["labels"] ,0.3)

In [103]:
from transformers import BertTokenizer, BertModel

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
bert_model.to(device)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0): BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          

In [104]:
lengths = whole_dataset["train"].map(
    lambda x: 
    {"tok" : bert_tokenizer(x["Premise"])}).map(lambda x:{"len": len(x["tok"]["token_type_ids"])})
print(lengths["len"])


  0%|          | 0/4314 [00:00<?, ?ex/s]

  0%|          | 0/4314 [00:00<?, ?ex/s]

tensor([29, 11, 29,  ..., 25, 36, 34])


In [105]:
print(np.quantile(lengths["len"], .9))
print(np.mean(lengths["len"].numpy()))

42.0
26.875289754288364


In [106]:
max_words = 50
embed_len = 768
batch_size = 32

# collate function that uses the tokenizer relative to the bert pretrained model
def bert_vectorize_batch(batch):
    X = [elem["Premise"] for elem in batch] # added the special Bert input token [CLS] in front of every premise
                                                       # it could be useful to add [SEP] and the consequences and check how BERT
                                                       # performs
    Y = [elem["labels"] for elem in batch]
    X = bert_tokenizer(X, padding="max_length", truncation="longest_first", return_tensors = "pt", max_length = max_words) 
    Y_tensor = torch.zeros(len(batch), Y[0].shape[0])
    for i, tokens in enumerate(Y):    
        Y_tensor[i] = Y[i]
    X_tensor = torch.stack([X["input_ids"], X["token_type_ids"], X["attention_mask"]])

    return X_tensor, Y_tensor

str_to_test = "Ammaccabanane"

train_dataset = whole_dataset["train"].remove_columns(["Stance", "Conclusion"])
val_dataset = whole_dataset["val"].remove_columns(["Stance", "Conclusion"])
test_dataset = whole_dataset["test"].remove_columns(["Stance", "Conclusion"])

# Construction of the Dataloaders for train and validation
bert_train_loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=lambda x:tuple(y.to(device) for y in bert_vectorize_batch(x)))
bert_val_loader  = DataLoader(val_dataset, batch_size=batch_size, collate_fn=lambda x:tuple(y.to(device) for y in bert_vectorize_batch(x)))
bert_test_loader  = DataLoader(test_dataset, batch_size=batch_size, collate_fn=lambda x:tuple(y.to(device) for y in bert_vectorize_batch(x)))

In [25]:
str_to_test = "Ammaccabanane"
str_to_test_2 = "Ammaccamele"

bert_model_2 = BertModel.from_pretrained('bert-base-uncased')
bert_model_2.to(device)

#normalized_str_input_ids = add_zeros(bert_tokenizer(str_to_test))["input_ids"]
#normalized_str_token_type_ids = add_zeros(bert_tokenizer(str_to_test))["token_type_ids"]
#normalized_str_att_mask = add_zeros(bert_tokenizer(str_to_test))["attention_mask"]#

#normalized_str_input_ids_2 = add_zeros(bert_tokenizer(str_to_test_2))["input_ids"]
#normalized_str_att_mask_2 = add_zeros(bert_tokenizer(str_to_test_2))["attention_mask"]

#torch_tensor_input_ids = torch.tensor(np.array([normalized_str_input_ids, normalized_str_input_ids_2]))
#torch_tensor_att_masks = torch.tensor(np.array([normalized_str_att_mask, normalized_str_att_mask_2]))

#torch_tensor_to_pass_3 = torch.tensor(np.array([torch_tensor_to_pass.numpy(), torch_tensor_to_pass_2.numpy()]))

normalized_batch = bert_tokenizer([str_to_test, str_to_test_2], padding="max_length", truncation="longest_first", return_tensors = "pt")
tensor_batch = torch.tensor(np.array([normalized_batch["input_ids"].numpy(), normalized_batch["token_type_ids"].numpy(), normalized_batch["attention_mask"].numpy()]))
#tensor_batch.to(device)

#to_pass = [torch_tensor_input_ids, torch_tensor_att_masks]

#print(torch_tensor_input_ids)
#print(torch_tensor_att_masks)


#X_batch_dict = {"input_ids":torch_tensor_input_ids, "attention_mask":torch_tensor_att_masks}
#X_batch_dict_2 = torch.tensor(np.array([torch_tensor_input_ids.numpy(), 
#                                       torch_tensor_att_masks.numpy()]))
#X_batch_dict_2.to(device)
#print(type(X_batch_dict_2))

# the output of the bert base model is the last hidden state, which represents 
# an embedding of the input (not exactly, but for now we can imagine it 
# as something similar to the output of the pretrained GloVe embedding),
# and the pooler_output, which is the last layer hidden-state of the first 
# token of the sequence 
print(bert_model_2(input_ids = tensor_batch[0].to(device), token_type_ids =  tensor_batch[1].to(device), attention_mask = tensor_batch[2].to(device)).pooler_output.shape)
print(bert_model_2(input_ids = tensor_batch[0].to(device), token_type_ids =  tensor_batch[1].to(device), attention_mask = tensor_batch[2].to(device)).pooler_output)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


torch.Size([2, 768])
tensor([[-0.9045, -0.3669, -0.4819,  ..., -0.5817, -0.6551,  0.8762],
        [-0.9257, -0.3762,  0.1518,  ..., -0.1092, -0.6157,  0.8963]],
       device='cuda:0', grad_fn=<TanhBackward0>)


In [113]:
num_classes = 20

# Simple model to perform some tests with pytorch
class PreBertBaseline(nn.Module):
    def __init__(self):
        super(PreBertBaseline, self).__init__() 
        # Not sure about this
        #self.seq_length = batch_size
        self.pretrained_bert = bert_model
        for param in self.pretrained_bert.parameters():
            param.requires_grad = False

        self.gru = nn.GRU(input_size = 768,
                          hidden_size = 100,
                          num_layers = 1,
                          batch_first=True, 
                          bidirectional = True)
        self.flatten = nn.Flatten(start_dim=1)
        self.linear_1 = nn.Linear(50*100*2, 3200)
        self.relu = nn.ReLU()

        self.linear_2 = nn.Linear(3200, 800)
        #nn.ReLU(),

        self.linear_3 = nn.Linear(800, 256)
        #nn.ReLU(),

        self.linear_4 = nn.Linear(256, 64)
        self.linear_5 = nn.Linear(64, num_classes)
        
                

    def forward(self, X_batch):
        #h0 = torch.zeros(2,X_batch.shape[0], embed_len)
        #h0 = h0.to(device)
        #out, hn = self.gru(X_batch, h0)
        #print(X_batch)
        #print(X_batch.shape)
        out = self.pretrained_bert(input_ids=X_batch[0], token_type_ids = X_batch[1], attention_mask = X_batch[2]).last_hidden_state
        h0 = torch.zeros(2, X_batch.shape[1], 100)
        h0 = h0.to(device)
        out, _ = self.gru(out, h0)
        out = self.flatten(out)
        out = self.linear_1(out)
        out = self.relu(out)
        out = self.linear_2(out)
        out = self.relu(out)
        out = self.linear_3(out)
        out = self.relu(out)
        out = self.linear_4(out)
        out = self.relu(out)
        out = self.linear_5(out)
        return out

# Function needed to compute the validation loss and the accuracy
def CalcValLossAndAccuracy(model, loss_fn, val_loader):
    with torch.no_grad():
      Y_shuffled, Y_preds, losses = [],[],[]
      for X, Y in val_loader:
        preds = model(X)
        loss = loss_fn(preds, Y)
        losses.append(loss.item())
        Y_shuffled.append(Y)
        Y_preds.append(preds.argmax(dim=-1))

      Y_shuffled = torch.cat(Y_shuffled)
      Y_preds = torch.cat(Y_preds)

      loss = torch.tensor(losses).mean()
      print("Valid Loss : {:.3f}".format(loss))
    return loss
    # print("Valid Acc  : {:.3f}".format(accuracy_score(Y_shuffled.detach().cpu().numpy(), Y_preds.detach().cpu().numpy())))

# Training function
def TrainModel(model, loss_fn, optimizer, train_loader, val_loader, epochs, early_stopping_info):
    patience_acc = 0
    precedent_loss = np.Inf
    model.train()
    for i in range(1, epochs+1):
        losses = []
        for X, Y in tqdm(train_loader):
            Y_preds = model(X)

            loss = loss_fn(Y_preds, Y)
            losses.append(loss.item())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        loss = CalcValLossAndAccuracy(model, loss_fn, val_loader)
        if precedent_loss - loss < early_stopping_info["delta"]:
           patience_acc = patience_acc + 1
        else:
          patience_acc = 0
          torch.save(model, "best.pth")

        if patience_acc > early_stopping_info["patience"]:
          return torch.load("best.pth")
        precedent_loss = loss

        if i%1==0:
            print("Train Loss : {:.3f}".format(torch.tensor(losses).mean()))
    return model

In [114]:
from torchinfo import summary
epochs = 50
learning_rate = 1e-4

loss_fn = nn.BCEWithLogitsLoss()
prebert_classifier = PreBertBaseline()
optimizer = Adam(prebert_classifier.parameters(), lr=learning_rate)

prebert_classifier.to(device)
summary(prebert_classifier, input_size=(3,1, max_words), 
        device="cuda", dtypes = [torch.int]*3)

Layer (type:depth-idx)                                  Output Shape              Param #
PreBertBaseline                                         [1, 20]                   --
├─BertModel: 1-1                                        [1, 768]                  --
│    └─BertEmbeddings: 2-1                              [1, 50, 768]              --
│    │    └─Embedding: 3-1                              [1, 50, 768]              (23,440,896)
│    │    └─Embedding: 3-2                              [1, 50, 768]              (1,536)
│    │    └─Embedding: 3-3                              [1, 50, 768]              (393,216)
│    │    └─LayerNorm: 3-4                              [1, 50, 768]              (1,536)
│    │    └─Dropout: 3-5                                [1, 50, 768]              --
│    └─BertEncoder: 2-2                                 [1, 50, 768]              --
│    │    └─ModuleList: 3-6                             --                        (85,054,464)
│    └─BertPooler: 2-3 

In [115]:
fix_random(seed)
prebert_classifier = TrainModel(prebert_classifier, loss_fn, optimizer, bert_train_loader, bert_val_loader, epochs, {"patience": 3, "delta": 1e-4})

100%|██████████| 135/135 [00:14<00:00,  9.28it/s]


Valid Loss : 0.394
Train Loss : 0.437


100%|██████████| 135/135 [00:14<00:00,  9.07it/s]


Valid Loss : 0.372
Train Loss : 0.388


100%|██████████| 135/135 [00:15<00:00,  8.78it/s]


...

Valid Loss : 0.340
Train Loss : 0.287


100%|██████████| 135/135 [00:15<00:00,  8.64it/s]


Valid Loss : 0.344
Train Loss : 0.271


100%|██████████| 135/135 [00:15<00:00,  8.78it/s]


Valid Loss : 0.353


In [122]:
print_report(prebert_classifier, bert_val_loader,val_dataset["labels"] ,0.3)

----- MACRO AVG. -----
  F1-score:	0.3846
  Precision:	0.397
  Recall:	0.4033
  Accuracy:	0.8355
----- PER-CLASS VALUES -----
  				F1-score	Precision	Recall		Accuracy	Support
  Self-direction: thought   	0.5969		0.5792		0.6158		0.8536		190
  Self-direction: action    	0.5324		0.5153		0.5507		0.7525		276
  Stimulation               	0.0816		0.3333		0.0465		0.9583		43
  Hedonism                  	0.1364		0.2727		0.0909		0.9648		33
  Achievement               	0.6229		0.5707		0.6855		0.7553		318
  Power: dominance          	0.386		0.3642		0.4104		0.8378		134
  Power: resources          	0.4135		0.43		0.3981		0.8869		108
  Face                      	0.0		0.0		0.0		0.9398		63
  Security: personal        	0.6688		0.5552		0.8408		0.709		377
  Security: societal        	0.677		0.5717		0.8299		0.7498		341
  Tradition                 	0.3822		0.4348		0.3409		0.9101		88
  Conformity: rules         	0.4511		0.4484		0.4538		0.7451		249
  Conformity: interpersonal 	0.0		0.0		0.0		0.9648		38
  Humil